In [1]:
import polars as pl
import numpy as np
import pandas as pd

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download("xvniyusheng/mindsmall-train")

In [5]:
print(path)
print(os.listdir(path))

C:\Users\HONOR\.cache\kagglehub\datasets\xvniyusheng\mindsmall-train\versions\1
['behaviors.tsv', 'entity_embedding.vec', 'news.tsv', 'relation_embedding.vec']


In [6]:
from pathlib import Path

path = Path(path)

In [7]:
# на странице датасета такие колонки

behaviors = pl.read_csv(
    path / "behaviors.tsv",
    separator="\t",
    has_header=False,
    new_columns=['impression_id', 'user_id', 'time', 'history', 'impressions'],
    quote_char=None
)

news = pl.read_csv(
    path / "news.tsv",
    separator="\t",
    has_header=False,
    new_columns=['id', 'category', 'subcategory', 'title', 'abstract', 'url',
                  'title_entities', 'abstract_entities'],
    quote_char=None
)

print(behaviors.shape)
print(news.shape)

(156965, 5)
(51282, 8)


In [8]:
behaviors.head(5)

impression_id,user_id,time,history,impressions
i64,str,str,str,str
1,"""U13740""","""11/11/2019 9:05:58 AM""","""N55189 N42782 N34694 N45794 N1…","""N55689-1 N35729-0"""
2,"""U91836""","""11/12/2019 6:11:30 PM""","""N31739 N6072 N63045 N23979 N35…","""N20678-0 N39317-0 N58114-0 N20…"
3,"""U73700""","""11/14/2019 7:01:48 AM""","""N10732 N25792 N7563 N21087 N41…","""N50014-0 N23877-0 N35389-0 N49…"
4,"""U34670""","""11/11/2019 5:28:05 AM""","""N45729 N2203 N871 N53880 N4137…","""N35729-0 N33632-0 N49685-1 N27…"
5,"""U8125""","""11/12/2019 4:11:21 PM""","""N10078 N56514 N14904 N33740""","""N39985-0 N36050-0 N16096-0 N84…"


In [9]:
news.head(5)

id,category,subcategory,title,abstract,url,title_entities,abstract_entities
str,str,str,str,str,str,str,str
"""N55528""","""lifestyle""","""lifestyleroyals""","""The Brands Queen Elizabeth, Pr…","""Shop the notebooks, jackets, a…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Prince Philip, Duk…","""[]"""
"""N19639""","""health""","""weightloss""","""50 Worst Habits For Belly Fat""","""These seemingly harmless habit…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Adipose tissue"", ""…","""[{""Label"": ""Adipose tissue"", ""…"
"""N61837""","""news""","""newsworld""","""The Cost of Trump's Aid Freeze…","""Lt. Ivan Molchanets peeked ove…","""https://assets.msn.com/labs/mi…","""[]""","""[{""Label"": ""Ukraine"", ""Type"": …"
"""N53526""","""health""","""voices""","""I Was An NBA Wife. Here's How …","""I felt like I was a fraud, and…","""https://assets.msn.com/labs/mi…","""[]""","""[{""Label"": ""National Basketbal…"
"""N38324""","""health""","""medical""","""How to Get Rid of Skin Tags, A…","""They seem harmless, but there'…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Skin tag"", ""Type"":…","""[{""Label"": ""Skin tag"", ""Type"":…"


In [10]:
interactions = (
    behaviors
    .filter(pl.col("history").is_not_null())
    .with_columns(history=pl.col("history").str.split(" "))
    .filter(pl.col("history").list.len() > 0)  
    # разложил для twotower 
    # да и labelencoder все равно не схавал бы список из итемов нормально
    .explode("history") 
    .rename({"history" : "item_id"})
    .with_columns(timestamp=pl.int_range(pl.len()).over("user_id"))
    .select(["user_id", "item_id", "timestamp"])
)

# TODO:
# спросить по архитектуре и кагл

print(interactions.head(20))
print(interactions.shape)

C:\Users\HONOR\AppData\Local\Temp\ipykernel_30296\1606026910.py:8: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("history")


shape: (20, 3)
┌─────────┬─────────┬───────────┐
│ user_id ┆ item_id ┆ timestamp │
│ ---     ┆ ---     ┆ ---       │
│ str     ┆ str     ┆ i64       │
╞═════════╪═════════╪═══════════╡
│ U13740  ┆ N55189  ┆ 0         │
│ U13740  ┆ N42782  ┆ 1         │
│ U13740  ┆ N34694  ┆ 2         │
│ U13740  ┆ N45794  ┆ 3         │
│ U13740  ┆ N18445  ┆ 4         │
│ …       ┆ …       ┆ …         │
│ U91836  ┆ N8129   ┆ 6         │
│ U91836  ┆ N1569   ┆ 7         │
│ U91836  ┆ N17686  ┆ 8         │
│ U91836  ┆ N13008  ┆ 9         │
│ U91836  ┆ N21623  ┆ 10        │
└─────────┴─────────┴───────────┘
(5107639, 3)


In [11]:
print("Users:", interactions["user_id"].n_unique())
print("Items:", interactions["item_id"].n_unique())

# числа те же что и на пандасе все норм

print(interactions.group_by("user_id").len().describe())

Users: 49108
Items: 33195
shape: (9, 3)
┌────────────┬─────────┬────────────┐
│ statistic  ┆ user_id ┆ len        │
│ ---        ┆ ---     ┆ ---        │
│ str        ┆ str     ┆ f64        │
╞════════════╪═════════╪════════════╡
│ count      ┆ 49108   ┆ 49108.0    │
│ null_count ┆ 0       ┆ 0.0        │
│ mean       ┆ null    ┆ 104.008288 │
│ std        ┆ null    ┆ 349.703277 │
│ min        ┆ U100    ┆ 1.0        │
│ 25%        ┆ null    ┆ 8.0        │
│ 50%        ┆ null    ┆ 20.0       │
│ 75%        ┆ null    ┆ 70.0       │
│ max        ┆ U9999   ┆ 18972.0    │
└────────────┴─────────┴────────────┘


In [74]:
# посмотрим на историю одного пользователя
user = behaviors["user_id"][0]

(
    behaviors
    .filter(pl.col("user_id") == user)
    .sort("time")
    .select([
        "impression_id",
        "user_id",
        "time",
        "history",
        "impressions"
    ])
    .head(10)
)
# TODO:
# истории связанные у пользователей значит при explode мы дублируем итемы которые уже были в истории, надо исправить
# одна из идей: брать последний impression как test, validate и train так и оставить + убедиться что у пользователей целостная история
# а нет отдельных

impression_id,user_id,time,history,impressions
i64,str,str,str,str
1,"""U13740""","""11/11/2019 9:05:58 AM""","""N55189 N42782 N34694 N45794 N1…","""N55689-1 N35729-0"""
154837,"""U13740""","""11/13/2019 3:27:40 PM""","""N55189 N42782 N34694 N45794 N1…","""N13907-0 N8509-0 N47061-0 N510…"
35263,"""U13740""","""11/9/2019 5:59:43 AM""","""N55189 N42782 N34694 N45794 N1…","""N20020-0 N3737-0 N43202-0 N187…"


In [ ]:
# проверяем количество повторов в таблице
(
    interactions
    .group_by(["user_id", "item_id"])
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
    .head(20)
)

user_id,item_id,count
str,str,u32
"""U32146""","""N42781""",310
"""U32146""","""N8148""",310
"""U70493""","""N47020""",261
"""U19040""","""N18870""",224
"""U38326""","""N61864""",198
…,…,…
"""U20271""","""N47020""",169
"""U80573""","""N20413""",168
"""U89995""","""N47020""",160


### encoding

In [12]:
from replay.preprocessing import LabelEncoder
from replay.preprocessing import LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)

interactions_encoded = encoder.fit_transform(interactions)

interactions_encoded.head()

c:\Users\HONOR\projects_hse\coursework\feature_enrichment\.venv\Lib\site-packages\pyspark\sql\udf.py:135: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


timestamp,user_id,item_id
i64,i64,i64
0,2201,25615
1,2201,18583
2,2201,14002
3,2201,20321
4,2201,4775


In [13]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True

    # TODO:
    # пока дропаю колд

)

train, validation = splitter.split(interactions_encoded) # плоский train


In [14]:
interactions_encoded.filter(pl.col("user_id") == 3).sort("timestamp")

timestamp,user_id,item_id
i64,i64,i64
0,3,16513
1,3,9482
2,3,12179
3,3,1576
4,3,28911
…,…,…
11,3,1576
12,3,28911
13,3,17964


In [15]:
print(train.shape)
print(validation.shape)

print(train.head())
print(validation.head())

(5058531, 3)
(48769, 3)
shape: (5, 3)
┌───────────┬─────────┬─────────┐
│ timestamp ┆ user_id ┆ item_id │
│ ---       ┆ ---     ┆ ---     │
│ i64       ┆ i64     ┆ i64     │
╞═══════════╪═════════╪═════════╡
│ 0         ┆ 2201    ┆ 25615   │
│ 0         ┆ 47474   ┆ 12337   │
│ 0         ┆ 36880   ┆ 410     │
│ 0         ┆ 14380   ┆ 20286   │
│ 0         ┆ 41327   ┆ 46      │
└───────────┴─────────┴─────────┘
shape: (5, 3)
┌───────────┬─────────┬─────────┐
│ timestamp ┆ user_id ┆ item_id │
│ ---       ┆ ---     ┆ ---     │
│ i64       ┆ i64     ┆ i64     │
╞═══════════╪═════════╪═════════╡
│ 1         ┆ 36783   ┆ 3514    │
│ 1         ┆ 43163   ┆ 4291    │
│ 1         ┆ 14156   ┆ 27611   │
│ 1         ┆ 5003    ┆ 14906   │
│ 1         ┆ 45954   ┆ 5686    │
└───────────┴─────────┴─────────┘


### нам еще нужен train для bert4rec и sasrec а там внутри полей списки итемов

In [16]:
train_sequences = (
    train
    .sort(["user_id", "timestamp"])
    .group_by("user_id", maintain_order=True)
    .agg(
        item_history=pl.col("item_id"),
        timestamp=pl.col("timestamp")
    )
)
train_sequences.head()

user_id,item_history,timestamp
i64,list[i64],list[i64]
0,"[5779, 13606, … 23635]","[0, 1, … 8]"
1,"[11166, 4458, … 4458]","[0, 1, … 7]"
2,"[21520, 23636, … 20685]","[0, 1, … 28]"
3,"[16513, 9482, … 23156]","[0, 1, … 14]"
4,"[7962, 11464, … 7796]","[0, 1, … 21]"


In [17]:
train_sequences.filter(pl.col("user_id") == 3).sort("timestamp")

user_id,item_history,timestamp
i64,list[i64],list[i64]
3,"[16513, 9482, … 23156]","[0, 1, … 14]"


In [18]:
train.filter(pl.col("user_id") == 3).sort("timestamp")

timestamp,user_id,item_id
i64,i64,i64
0,3,16513
1,3,9482
2,3,12179
3,3,1576
4,3,28911
…,…,…
10,3,12179
11,3,1576
12,3,28911


### все норм разделилось, порядок в истории сохранился

## Обучение

In [19]:
%pip install pytorch-lightning lightning torch

Note: you may need to restart the kernel to use updated packages.


### потом понадобится ну а вообще надо тензор схему

In [32]:
from replay.data import Dataset, FeatureSchema, FeatureInfo, FeatureType, FeatureHint, FeatureSource

schema = FeatureSchema([
    FeatureInfo(
        column="user_id",
        feature_type=FeatureType.CATEGORICAL,
        feature_hint=FeatureHint.QUERY_ID,  # query_id в replay заменяет понятие user_id
    ),
    FeatureInfo(
        column="item_id",
        feature_type=FeatureType.CATEGORICAL,
        feature_hint=FeatureHint.ITEM_ID,
    ),
    FeatureInfo(
        column="timestamp",
        feature_type=FeatureType.NUMERICAL,
        feature_hint=FeatureHint.TIMESTAMP,
    ),
])

train_dataset = Dataset(
    interactions=train,
    feature_schema=schema
)

validation_dataset = Dataset(
    interactions=validation,
    feature_schema=schema
)

In [34]:
import torch
print(torch.__version__)

from replay.utils import TORCH_AVAILABLE
print("TORCH_AVAILABLE =", TORCH_AVAILABLE)

2.15.0+cpu
TORCH_AVAILABLE = ModuleSpec(name='lightning', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000002B53ED50AD0>, origin='c:\\Users\\HONOR\\projects_hse\\coursework\\feature_enrichment\\.venv\\Lib\\site-packages\\lightning\\__init__.py', submodule_search_locations=['c:\\Users\\HONOR\\projects_hse\\coursework\\feature_enrichment\\.venv\\Lib\\site-packages\\lightning'])


### для низкоуровневых моделей

In [75]:
from replay.data.nn import TensorSchema, TensorFeatureInfo, TensorFeatureSource

num_all_items = len(interactions_encoded["item_id"].unique())

tensor_schema = TensorSchema([
    TensorFeatureInfo(
        name="item_id",
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        embedding_dim=128,
        padding_value=num_all_items,
        cardinality=num_all_items,
        feature_hint=FeatureHint.ITEM_ID,
        feature_sources=[
            TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")
        ]
    ),
])


### sequence dataset'ы устарели поэтому через паркет модуль

In [83]:
train_sequences_for_nn = train_sequences.select(
    "user_id",
    pl.col("item_history").alias("item_id"),
)

train_sequences_for_nn.write_parquet("data/train_sequences.parquet")
train_sequences_for_nn.head()

user_id,item_id
i64,list[i64]
0,"[5779, 13606, … 23635]"
1,"[11166, 4458, … 4458]"
2,"[21520, 23636, … 20685]"
3,"[16513, 9482, … 23156]"
4,"[7962, 11464, … 7796]"


## sasrec parquetmodule

In [ ]:
from replay.data.nn import ParquetModule
from replay.nn.transform.template import make_default_sasrec_transforms, make_default_twotower_transforms 


# общие для разных моделей
MAX_LEN = 120
BATCH_SIZE = 64

metadata = {
    "train": {
        "item_id": {
            # NextTokenTransform shifts labels by 1 so read one extra item
            "shape": MAX_LEN + 1,
            # RePlay 0.23.0 expects the key "padding" (not "padding_value"
            "padding": num_all_items,
        }
    }
}

datamodule_sasrec = ParquetModule(
    batch_size=BATCH_SIZE,
    train_path="data/train_sequences.parquet",
    metadata=metadata,
    transforms=make_default_sasrec_transforms(tensor_schema),
)


C:\Users\HONOR\AppData\Local\Temp\ipykernel_30296\2597564336.py:18: UserWarning: The following dataset paths aren't provided: validate,test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  datamodule_sasrec = ParquetModule(


## bert4rec parquetmodule

In [ ]:
from torch.utils.data import DataLoader
from replay.data.nn import PandasSequentialDataset
from replay.models.nn.sequential.bert4rec import Bert4RecTrainingDataset

BERT_PAD_ID = num_all_items

bert_tensor_schema = TensorSchema([
    TensorFeatureInfo(
        name="item_id",
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        embedding_dim=128,
        padding_value=BERT_PAD_ID,
        cardinality=num_all_items + 1,
        feature_hint=FeatureHint.ITEM_ID,
        feature_sources=[
            TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")
        ],
    ),
])

bert_sequences_pd = train_sequences_for_nn.to_pandas()
bert_sequential = PandasSequentialDataset(
    tensor_schema=bert_tensor_schema,
    query_id_column="user_id",
    item_id_column="item_id",
    sequences=bert_sequences_pd,
)

bert_train_dataset = Bert4RecTrainingDataset(
    sequential=bert_sequential,
    max_sequence_length=MAX_LEN,
    mask_prob=0.15,
    padding_value=BERT_PAD_ID,
)
bert_train_loader = DataLoader(
    bert_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

C:\Users\HONOR\AppData\Local\Temp\ipykernel_30296\3962192486.py:25: DeprecationWarning: `PandasSequentialDataset` class is deprecated.
  bert_sequential = PandasSequentialDataset(


## sasrec

In [ ]:
from lightning import Trainer
from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule as RePlayLightningModule

# обертка позволяет использовать lightning train
model_sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=128,
    num_heads=4,
    num_blocks=2,
    max_sequence_length=MAX_LEN,
    dropout=0.2,
)
sasrec_module = RePlayLightningModule(model=model_sasrec)

trainer_sasrec = Trainer(
    max_epochs=1,
    accelerator="auto", # у меня на cpu
    devices=1,
    limit_val_batches=0,       # this notebook currently supplies train data only to ParquetModule
    num_sanity_val_steps=0,
    enable_checkpointing=False,
    logger=False,
)

trainer_sasrec.fit(sasrec_module, datamodule=datamodule_sasrec)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name  | Type   | Params | Mode  | FLOPs
-------------------------------------------------
0 | model | SasRec | 4.5 M  | train | 0    
-------------------------------------------------
4.5 M     Trainable params
0         Non-trainable params
4.5 M     Total params
17.855    Total estimated model params size (MB)
40        Modules in train mode
0         Modules in eval mode
0         Total Flops
c:\Users\HONOR\projects_hse\coursework\feature_enrichment\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\HONOR\projects_hse\coursework\feature_en

Epoch 1:   1%|▏         | 10/765 [00:18<22:50,  0.55it/s, learning_rate_step=0.001, train_loss_step=6.920, learning_rate_epoch=0.001, train_loss_epoch=7.980] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\HONOR\projects_hse\coursework\feature_enrichment\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Epoch 0:   0%|          | 0/765 [36:57<?, ?it/s]


## обучение bert

In [ ]:
from lightning import Trainer
from replay.models.nn.sequential import Bert4Rec

model_bert4rec = Bert4Rec(
    tensor_schema=bert_tensor_schema,
    block_count=2,
    head_count=4,
     # должен соответствовать тензон схеме embedding_dim=128
    hidden_size=128,
    max_seq_len=MAX_LEN,
    dropout_rate=0.1,
)

trainer_bert4rec = Trainer(
    max_epochs=1,
    accelerator="auto",
    devices=1,
    enable_checkpointing=False,
    logger=False,
)

# Lightning-compatible
trainer_bert4rec.fit(
    model_bert4rec,
    train_dataloaders=bert_train_loader,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name   | Type             | Params | Mode  | FLOPs
------------------------------------------------------------
0 | _model | Bert4RecModel    | 8.9 M  | train | 0    
1 | _loss  | CrossEntropyLoss | 0      | train | 0    
------------------------------------------------------------
8.9 M     Trainable params
0         Non-trainable params
8.9 M     Total params
35.774    Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0:   1%|          | 4/765 [00:07<22:42,  0.56it/s, train_loss_step=10.30]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\HONOR\projects_hse\coursework\feature_enrichment\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# хз пока понадобится ли

from replay.nn.transform import (
    TrimTransform,
    CopyTransform,
    TokenMaskTransform,
)

transforms = {
    "train": [
        TrimTransform(seq_len=MAX_LEN, feature_names=["item_history"]),
        # Копируем оригинал в labels для расчета лосса
        CopyTransform({"item_history": "labels"}),
        # Создаем маску для MLM (аналог BERT)
        TokenMaskTransform(
            token_name="item_history", 
            mask_prob=0.15
        )
    ],
    "validate": [
        TrimTransform(seq_len=MAX_LEN, feature_names=["item_history"]),
        # Для валидации обычно маскируют последний элемент, 
        # но если встроенной валидации нет, можно оставить только Trim
    ]
}

datamodule_bert4rec = ParquetModule(
    batch_size=BATCH_SIZE,
    transforms=transforms,
    metadata=metadata,
    train_path="train_sequences.parquet"
)

C:\Users\HONOR\AppData\Local\Temp\ipykernel_30296\3950219493.py:25: UserWarning: The following dataset paths aren't provided: validate,test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  datamodule_bert4rec = ParquetModule(


## TODO:
- добавить в схему validation test
- добавить callback и графы train_loss логи крч
- построить метрики на предикте
- понять зачем считать несколько эпох
- мб добавить кэш или сразу на кагл свалить